In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob

In [ ]:
varn_S_list = ["C5H7O2N(aq)", "CH2O(aq)", "CO2(aq)", "N2(aq)", "NH4+", "NO2-", "NO3-", "O2(aq)"]

def plot_scenarios_comparison(varn_S, flag_save=False, xlim=None, ylim=None):
    """
    Creates comparison plots for a specific variable with enhanced visibility.
    Parameters:
    varn_S (str): The variable name to plot (e.g., "CH2O(aq)")
    xlim (tuple or list): Optional x-axis limits. Can be:
        - Single tuple (min, max): Applied to all subplots
        - List of 3 tuples [(min1, max1), (min2, max2), (min3, max3)]: Applied to each subplot separately
        - None: Auto-scaling for all subplots
    ylim (tuple or list): Optional y-axis limits. Can be:
        - Single tuple (min, max): Applied to all subplots
        - List of 3 tuples [(min1, max1), (min2, max2), (min3, max3)]: Applied to each subplot separately
        - None: Auto-scaling for all subplots
    """
    # Define scenarios with distinct styling
    scenarios_info = [
        {'folder': 'casecybernetic-run1.s1', 'label': 'Scenario 1', 'color': 'blue', 'linestyle': '-', 'marker': 'o', 'alpha': 0.8},
        {'folder': 'casecybernetic-run1.s2', 'label': 'Scenario 2', 'color': 'red', 'linestyle': '--', 'marker': 's', 'alpha': 0.8},
        {'folder': 'casecybernetic-run1.s3', 'label': 'Scenario 3', 'color': 'green', 'linestyle': '-.', 'marker': '^', 'alpha': 0.8},
        {'folder': 'casecybernetic-run1.s4', 'label': 'Scenario 4', 'color': 'orange', 'linestyle': ':', 'marker': 'D', 'alpha': 0.8}
    ]
    
    # Create output directory
    output_dir = './figs_scenarios_comparison'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Check if varn_S is in the list and get its index
    if varn_S not in varn_S_list:
        print(f"Error: {varn_S} not found in variable list: {varn_S_list}")
        return
    
    i = varn_S_list.index(varn_S)
    print(f"Processing comparison plots for: {varn_S}")
    
    clean_varn = varn_S.replace('(', '').replace(')', '').replace('.', '_').replace('+', 'plus').replace('-', 'minus')
    
    # Process xlim and ylim parameters
    def process_limits(limits):
        """Process xlim or ylim parameter into a list of 3 tuples"""
        if limits is None:
            return [None, None, None]
        elif isinstance(limits, tuple) and len(limits) == 2:
            # Single tuple applied to all subplots
            return [limits, limits, limits]
        elif isinstance(limits, list) and len(limits) == 3:
            # List of 3 tuples for each subplot
            return limits
        else:
            print(f"Warning: Invalid limits format {limits}. Using auto-scaling.")
            return [None, None, None]
    
    xlims = process_limits(xlim)
    ylims = process_limits(ylim)
    
    # Create figure with 1x3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    data_found = False
    
    # Load data from each scenario
    for scenario_info in scenarios_info:
        csv_file = os.path.join(scenario_info['folder'], 'data_average_con_overtime',
                              f'timeseries_{i+1:02d}_{clean_varn}.csv')
        
        if os.path.exists(csv_file):
            try:
                df = pd.read_csv(csv_file)
                data_found = True
                
                # Subsample data for markers (every Nth point to avoid overcrowding)
                marker_every = max(1, len(df) // 20)  # Show markers every 20th point or so
                
                # Plot 1: Subsurface concentration (bulk volume)
                axes[0].plot(df['time_days'], df['subsurface_conc_bulk_vol'],
                           color=scenario_info['color'],
                           label=scenario_info['label'],
                           linestyle=scenario_info['linestyle'],
                           marker=scenario_info['marker'],
                           markevery=marker_every,
                           markersize=6,
                           linewidth=2.5,
                           alpha=scenario_info['alpha'])
                
                # Plot 2: Subsurface concentration (water volume)  
                axes[1].plot(df['time_days'], df['subsurface_conc_water_vol'],
                           color=scenario_info['color'],
                           label=scenario_info['label'],
                           linestyle=scenario_info['linestyle'],
                           marker=scenario_info['marker'],
                           markevery=marker_every,
                           markersize=6,
                           linewidth=2.5,
                           alpha=scenario_info['alpha'])
                
                # Plot 3: Surface concentration (water volume)
                axes[2].plot(df['time_days'], df['surface_conc_water_vol'],
                           color=scenario_info['color'],
                           label=scenario_info['label'],
                           linestyle=scenario_info['linestyle'],
                           marker=scenario_info['marker'],
                           markevery=marker_every,
                           markersize=6,
                           linewidth=2.5,
                           alpha=scenario_info['alpha'])
                
            except Exception as e:
                print(f"Error reading {csv_file}: {e}")
        else:
            print(f"File not found: {csv_file}")
    
    if not data_found:
        print(f"No data found for {varn_S}, skipping...")
        plt.close(fig)
        return
    
    # Customize all subplots
    subplot_titles = [
        f'Subsurface {varn_S}\n(mol/m³ bulk volume)',
        f'Subsurface {varn_S}\n(mol/m³ water volume)',
        f'Surface {varn_S}\n(mol/m³ water volume)'
    ]
    
    for j, ax in enumerate(axes):
        ax.set_xlabel('Time [d]', fontsize=12)
        ax.set_ylabel(f'{varn_S} concentration', fontsize=12)
        ax.set_title(subplot_titles[j], fontsize=13, fontweight='bold')
        ax.legend(fontsize=11, framealpha=0.9, loc='best')
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        ax.set_facecolor('#fafafa')  # Light background
        
        # Apply custom xlim and ylim for each subplot separately
        if xlims[j] is not None:
            ax.set_xlim(xlims[j])
        if ylims[j] is not None:
            ax.set_ylim(ylims[j])
    
    # Add main title
    fig.suptitle(f'Scenario Comparison: {varn_S}', fontsize=16, fontweight='bold', y=1.02)
    
    # Adjust layout and save
    plt.tight_layout()
    plt.show()
    
    # Save figure
    if flag_save:
        filename = os.path.join(output_dir, f'comparison_{i+1:02d}_{clean_varn}.png')
        fig.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Comparison plot saved to {filename}")
    
    plt.close(fig)

In [ ]:
# # Example 1: Same limits for all subplots
# plot_scenarios_comparison("CH2O(aq)", 
#                          xlim=(0, 100), 
#                          ylim=(0, 50),
#                          flag_save=True)

# # Example 2: Different limits for each subplot
# plot_scenarios_comparison("CO2(aq)", 
#                          xlim=[(0, 100), (10, 90), (0, 200)],  # Different x-limits for each subplot
#                          ylim=[(0, 50), (5, 45), (0, 80)],    # Different y-limits for each subplot
#                          flag_save=True)

# # Example 3: Mixed approach - same x-limits, different y-limits
# plot_scenarios_comparison("NH4+", 
#                          xlim=(0, 150),                        # Same x-limits for all
#                          ylim=[(0, 10), (0, 15), (0, 5)],     # Different y-limits for each
#                          flag_save=True)

# # Example 4: Auto-scaling (original behavior)
# plot_scenarios_comparison("N2(aq)", flag_save=True)

# # Example 5: Only set limits for specific subplots (use None for auto-scaling)
# plot_scenarios_comparison("O2(aq)", 
#                          xlim=[(0, 100), None, (20, 80)],     # Auto x-scale for middle subplot
#                          ylim=[None, (0, 30), None],          # Only set y-limits for middle subplot
#                          flag_save=True)

In [ ]:
# Define the variable list outside the function
for varn_S in varn_S_list:
        plot_scenarios_comparison(varn_S, flag_save=True)

In [ ]:
# To check specific substrate
plot_scenarios_comparison("NH4+", ylim=[(0.5e-13, 2e-13),(0.5e-12, 2e-12),(-0.1e-12, 3e-10)])

In [ ]:
plot_scenarios_comparison("NO3-", ylim=[(0.0032, 0.004),(0.03, 0.05),(0, 0.04)])

In [ ]:
plot_scenarios_comparison("C5H7O2N(aq)", ylim=[(0.0, 3e-5),(0.0, 3e-4),(-0.1e-3, 7e-3)])